# Getting data from preprint service


bioRxiv distributes each preprint as a `.meca` file which is a ZIP archive containing JATS XML, original figure images (PNG/TIFF), PDF and other supplemental files.

The project corpus is the full May 2026 monthly dump (7,132 articles, ~75 GB compressed).


## Bulk download via rclone from bioRxiv S3

bioRxiv hosts monthly MECA dumps in the public AWS S3 bucket `s3://biorxiv-src-monthly/`. The bucket is **requester-pays**. That requires an AWS account which will be charged for the egress. No IAM policy is required beyond a standard AWS account.

Recent months sit in `Current_Content/<Month>_<Year>/` (`Current_Content/May_2026/`). Older content is moved into batches under `Back_Content/`. Full bioRxiv corpus is more than a TB of compressed archives which will result if more than a few TB of images.

[bioRxiv: Machine access and text/data mining resources](https://www.biorxiv.org/tdm)

In [1]:
import os
# Verify rclone is installed
import shutil, subprocess
rclone = shutil.which("rclone")
print(f"rclone: {rclone or 'NOT FOUND'}")
if rclone:
    r = subprocess.run([rclone, "--version"], capture_output=True, text=True)
    print(r.stdout.split("\n")[0])

rclone: /usr/local/bin/rclone
rclone v1.75.0


In [12]:
"""
AWS credentials are required.
 Requester-pays buckets need AWS account with billing enabled, the transfer is charged to that account.
"""
import dotenv

dotenv.load_dotenv()

AWS_ACCESS_KEY_ID = os.environ["AWS_ACCESS_KEY_ID"]
AWS_SECRET_ACCESS_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]



In [29]:
%%bash


mkdir -p ~/.config/rclone
cat > ~/.config/rclone/rclone.conf << EOF

[biorxiv]
type = s3
provider = AWS
env_auth = false
access_key_id = $AWS_ACCESS_KEY_ID
secret_access_key = $AWS_SECRET_ACCESS_KEY
region = us-east-1
region = us-east-1
endpoint =
location_constraint =
acl = private
server_side_encryption =
storage_class =
ca_cert = ~/zscaler.pem

EOF

In [28]:
# list directories on the S3 bucket
import subprocess, shutil

rclone = shutil.which("rclone")

BUCKET_ROOT = "biorxiv:biorxiv-src-monthly/Current_Content"

if rclone:
    r = subprocess.run(
        [rclone, "lsd", BUCKET_ROOT, "--s3-requester-pays"],
        capture_output=True, text=True
    )
    print(r.stdout or "(no output)")
    if r.stderr:
        print("STDERR:", r.stderr[:400])
else:
    print("rclone not on PATH — install from https://rclone.org/downloads/")

(no output)
STDERR: 2026/09/28 11:47:19 ERROR : error listing: operation error S3: ListObjectsV2, https response error StatusCode: 403, RequestID: 6RHNCF4ENRNJ50K9, HostID: POgwHERyiEeYatx4idQJnOQXBSKIYKQaQRYdI0wFKQfqQ1jPGZWio6L4iY0STLmiNkJy189Wn+BjBDGASJaWHqgdr2O28c6x, api error InvalidAccessKeyId: The AWS Access Key Id you provided does not exist in our records.
2026/09/28 11:47:19 NOTICE: Failed to lsd with 2 erro


In [25]:
# Download one month
from pathlib import Path

MONTH = "May_2026"
DEST  = Path("../data") / MONTH
DEST.mkdir(parents=True, exist_ok=True)

S3_PATH = f"{BUCKET_ROOT}/{MONTH}/"

rclone_cmd = [
    rclone or "rclone",
    "sync", S3_PATH, str(DEST),
    "--s3-requester-pays",
    "--transfers", "8",
    "--progress",
    "--stats-one-line",
]

print("Command:")
print("  " + " ".join(rclone_cmd))
print(f"\nDestination: {DEST.resolve()}")

Command:
  /usr/local/bin/rclone sync biorxiv:biorxiv-src-monthly/Current_Content/May_2026/ ../data/May_2026 --s3-requester-pays --transfers 8 --progress --stats-one-line

Destination: /Users/kkrt923/Projects/cm3070-final/data/May_2026


In [26]:
#Run the download. ~75GB
import subprocess
r = subprocess.run(rclone_cmd, text=True)
print(f"exit code: {r.returncode}")


2026/09/28 11:46:31 ERROR : S3 bucket biorxiv-src-monthly path Current_Content/May_2026: error reading source root directory: operation error S3: ListObjectsV2, https response error StatusCode: 403, RequestID: A40P8EVNBYC14DJT, HostID: DymBhZeLVI/+xPMgqG2Sb0IWXKO8vzfv/oPcUHGZqtWS5a7+nueQUMZO6eZBFyVs+Gh7Bhd5jDM=, api error InvalidAccessKeyId: The AWS Access Key Id you provided does not exist in our records.
0 B / 0 B, -, 0 B/s, ETA -
2026/09/28 11:46:31 ERROR : Local file system at ．．/data/May_2026: not deleting files as there were IO errors
0 B / 0 B, -, 0 B/s, ETA -
2026/09/28 11:46:31 ERROR : Local file system at ．．/data/May_2026: not deleting directories as there were IO errors
0 B / 0 B, -, 0 B/s, ETA -
2026/09/28 11:46:31 ERROR : Attempt 1/3 failed with 1 errors and: operation error S3: ListObjectsV2, https response error StatusCode: 403, RequestID: A40P8EVNBYC14DJT, HostID: DymBhZeLVI/+xPMgqG2Sb0IWXKO8vzfv/oPcUHGZqtWS5a7+nueQUMZO6eZBFyVs+Gh7Bhd5jDM=, api error InvalidAccessKeyId

2026/09/28 11:46:31 NOTICE: Failed to sync: operation error S3: ListObjectsV2, https response error StatusCode: 403, RequestID: A40KYQ6H2HE21HXG, HostID: w7OCD+x/P/MTUOpc2LD1pQXPNoARVdXHxeUsiALYKCrJ0/DiLpPSqydN/4kWA9/I7VXNzesmKOo=, api error InvalidAccessKeyId: The AWS Access Key Id you provided does not exist in our records.
